# F6 — Week 12 Performance Review

**Objective**: Review the optimisation performance of F6 across all 10 submission rounds before deciding on a strategy for the next submission.

**Function**: F6 (5D input, 1D output, maximisation)

This notebook loads the Week 12 data, visualises convergence and input-space coverage, evaluates performance, and proposes strategy improvements. No optimisation loop is run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
import math

# ── Function Configuration ──
FUNC_NUM = 6
N_DIMS = 5
N_INITIAL = 20
WEEK = 11
USE_LOG_SCALE = False
DATA_DIR = '../../data/f6/'

## Step 1 — Load Data

In [ ]:
# Load Week 12 data
inputs = np.load(f'{DATA_DIR}updated_inputs - Week {WEEK}.npy')
outputs = np.load(f'{DATA_DIR}updated_outputs - Week {WEEK}.npy')

n_total = len(outputs)
n_submissions = n_total - N_INITIAL

print(f'F{FUNC_NUM} — Week {WEEK} Data Summary')
print(f'  Input dimensions:  {N_DIMS}')
print(f'  Total samples:     {n_total}')
print(f'  Initial samples:   {N_INITIAL}')
print(f'  Submissions:       {n_submissions}')
print(f'  Input shape:       {inputs.shape}')
print(f'  Output shape:      {outputs.shape}')
print(f'  Best output:       {outputs.max():.6g}')
print(f'  Worst output:      {outputs.min():.6g}')
print()

# Display data table
print('Sample | ' + ' | '.join([f'x{j+1:d}' for j in range(N_DIMS)]) + ' | y')
print('-' * (10 + N_DIMS * 12 + 15))
for i in range(n_total):
    label = 'init' if i < N_INITIAL else f'wk{i - N_INITIAL + 3}'
    row = f'{i+1:>4d}({label:>4s}) | '
    row += ' | '.join([f'{inputs[i, j]:.6f}' for j in range(N_DIMS)])
    row += f' | {outputs[i]:.6g}'
    print(row)

## Step 2 — Convergence Plot

Running best (maximum) objective value over all samples.

In [ ]:
# Compute running best (maximisation)
running_best = np.maximum.accumulate(outputs)

fig, ax = plt.subplots(figsize=(10, 5))

# Split into initial and submission regions
x_all = np.arange(1, n_total + 1)

if USE_LOG_SCALE:
    # Clamp non-positive values to epsilon before log
    plot_vals = np.where(running_best > 0, running_best, 1e-300)
    ax.set_yscale('log')
    ax.set_ylabel('Running Best (log scale)')
else:
    plot_vals = running_best
    ax.set_ylabel('Running Best')

# Plot initial samples in blue
ax.plot(x_all[:N_INITIAL], plot_vals[:N_INITIAL], 'o-', color='tab:blue',
        label='Initial samples', markersize=5)

# Plot submissions in orange
ax.plot(x_all[N_INITIAL-1:], plot_vals[N_INITIAL-1:], 'o-', color='tab:orange',
        label='Submissions', markersize=5)

# Vertical separator
ax.axvline(x=N_INITIAL + 0.5, color='grey', linestyle='--', alpha=0.7,
           label='Initial / Submission boundary')

ax.set_xlabel('Sample Number')
ax.set_title(f'F{FUNC_NUM} — Convergence Plot (Week {WEEK})')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Step 3 — 2D Pair Plots

Scatter plots of each unique pair of input dimensions showing spatial coverage. Initial samples in **blue** (unmarked), submission samples in **orange** (numbered by submission week).

In [ ]:
# Generate all unique pairs of input dimensions
pairs = list(combinations(range(N_DIMS), 2))
n_pairs = len(pairs)

if n_pairs == 0:
    print('Only 1 dimension — no pair plots to display.')
else:
    # Grid layout
    n_cols = min(n_pairs, 3) if n_pairs <= 6 else min(n_pairs, 5) if n_pairs <= 15 else 7
    n_rows = math.ceil(n_pairs / n_cols)
    fig_width = n_cols * 4
    fig_height = n_rows * 3.5

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height),
                             squeeze=False)

    for idx, (di, dj) in enumerate(pairs):
        row, col = divmod(idx, n_cols)
        ax = axes[row][col]

        # Initial samples — blue, unmarked
        ax.scatter(inputs[:N_INITIAL, di], inputs[:N_INITIAL, dj],
                   c='tab:blue', marker='o', s=40, alpha=0.7, label='Initial')

        # Submission samples — orange, numbered by week
        for k in range(N_INITIAL, n_total):
            week_num = k - N_INITIAL + 3  # Weeks start at 3
            ax.scatter(inputs[k, di], inputs[k, dj],
                       c='tab:orange', marker='o', s=40, alpha=0.7)
            ax.annotate(str(week_num), (inputs[k, di], inputs[k, dj]),
                        textcoords='offset points', xytext=(4, 4),
                        fontsize=7, color='tab:orange', fontweight='bold')

        ax.set_xlabel(f'x{di+1}')
        ax.set_ylabel(f'x{dj+1}')
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f'x{di+1} vs x{dj+1}')
        ax.grid(True, alpha=0.3)

    # Add legend to first subplot
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='tab:blue', label='Initial'),
                       Patch(facecolor='tab:orange', label='Submissions')]
    axes[0][0].legend(handles=legend_elements, loc='upper right', fontsize=7)

    # Hide empty subplots
    for idx in range(n_pairs, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    fig.suptitle(f'F{FUNC_NUM} — Input Space Coverage (Week {WEEK})', fontsize=14)
    fig.tight_layout()
    plt.show()

## Step 4 — Performance Evaluation

### Current Strategy (Week 9)

- **Surrogate**: SFGP Matérn-1.5 ARD, noise_lb=1e-2
- **Acquisition**: qLogNEI q=4, rank-based interior penalty
- **Key hyperparameters**: 50 restarts, 3000 raw samples, milk ≥ 0.10 constraint

### Performance Summary

In [ ]:
# Performance metrics
running_best = np.maximum.accumulate(outputs)
init_best = running_best[N_INITIAL - 1]

# Count improvements and detect stalling
improvements = 0
consec_no_improve = 0
max_consec_no_improve = 0
prev_best = init_best

for j in range(N_INITIAL, n_total):
    if running_best[j] > prev_best:
        improvements += 1
        consec_no_improve = 0
    else:
        consec_no_improve += 1
        max_consec_no_improve = max(max_consec_no_improve, consec_no_improve)
    prev_best = running_best[j]

stalling = max_consec_no_improve >= 3

print(f'Best value (initial):     {init_best:.6g}')
print(f'Best value (final):       {running_best[-1]:.6g}')
print(f'Improvements:             {improvements}/{n_submissions}')
print(f'Max consecutive no-improve: {max_consec_no_improve}')
print(f'Stalling (≥3 consec):     {stalling}')
print()

# Per-submission performance
print('Week | Output         | Best-so-far    | Improved?')
print('-' * 55)
for j in range(N_INITIAL, n_total):
    week_num = j - N_INITIAL + 3
    improved = '✓' if (j == N_INITIAL and outputs[j] > init_best) or \
               (j > N_INITIAL and running_best[j] > running_best[j-1]) else '✗'
    print(f'  {week_num:>2d} | {outputs[j]:>14.6g} | {running_best[j]:>14.6g} | {improved}')

### Evaluation

F6 is the **best-performing function** with **4 improvements** in 10 submission rounds and **no stalling** (maximum 2 consecutive non-improving). The best value moved from -0.7143 (initial) to -0.1115 — substantial progress towards zero in this all-negative landscape.

Key observations:
- 4/10 submissions improved the running best — the highest improvement rate across all functions
- Maximum only 2 consecutive non-improving submissions — the only function not flagged as stalling
- The rank-based interior penalty with milk ≥ 0.10 constraint is effectively balancing feasibility and optimality
- The negative outputs trending towards zero indicate the surrogate is correctly modelling the landscape
- 5D input space with 10 pair plots provides rich visualisation of the exploration pattern

**Stalling status**: NO — maximum 2 consecutive submissions without improvement (below threshold of 3).

## Step 5 — Proposed Strategy Improvements

F6 is the best performer (4/10 improvements, no stalling). Minor refinements to maintain progress:

1. **Maintain current approach** — The SFGP Matérn-1.5 ARD with rank-based interior penalty is working well. Major changes could disrupt the positive trajectory.

2. **Consider tightening the milk constraint** — If feasibility is well-established, the milk ≥ 0.10 threshold could be increased slightly (e.g., 0.12) to focus on higher-quality feasible regions.

3. **Reduce noise_lb from 1e-2 to 1e-3** — The relatively high noise floor may be smoothing out the posterior too much, preventing fine-grained local improvement near the current best.

4. **Review IP steepness parameter** — The penalty steepness controls how sharply infeasible regions are penalised. Fine-tuning this could improve the balance between exploration of the feasible boundary and interior exploitation.

5. **Increase raw samples to 5000** — With 5D input space and 30 total samples, the acquisition surface is complex. More seed points help find better candidates.

**Priority**: LOW — F6 is performing well. Changes should be conservative to avoid disrupting progress.

## Step 6 — Week 12 Optimisation Run

### Strategy Changes (Week 12)

1. **Noise floor: 1e-2 → 1e-3** — Tighter noise constraint allows the GP to fit more precisely while still preventing exact interpolation.
2. **Milk threshold: ≥0.10 → ≥0.12** — Domain knowledge suggests milk should be at least 12% of the recipe. Fallback to 0.10 if no feasible candidate with ≥0.12.
3. **Raw samples: 3000 → 5000** — More initial Sobol points for acquisition optimisation.
4. **Rank-based interior penalty** — Carried from week 9 (sign-invariant for negative outputs).

In [ ]:
# ── Imports & Configuration ───────────────────────────────────────────────────
import torch
import copy
import warnings
import re
from botorch.models import SingleTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.constraints import GreaterThan

# ── Hyperparameters ──
KERNEL_NU = 1.5
ARD_NUM_DIMS = 5
DIM = 5
NOISE_LB = 1e-3           # Tightened from 1e-2
N_MLL_RESTARTS = 15
MC_SAMPLES = 512
Q_BATCH = 4
NUM_RESTARTS = 50          # L-BFGS restarts for acquisition
RAW_SAMPLES = 5000         # Increased from 3000
STEEPNESS = 1.0
FLOOR = 0.01
MILK_THRESHOLD = 0.12      # Raised from 0.10 (fallback: 0.10)
MILK_FALLBACK = 0.10
GRID_RES = 80

ingredient_names = ['flour', 'sugar', 'eggs', 'butter', 'milk']

# Feasibility bounds: x4 (milk) ≥ MILK_THRESHOLD, others ≥ 0.01
BOUNDS = torch.tensor(
    [[0.01, 0.01, 0.01, 0.01, MILK_THRESHOLD],
     [1.00, 1.00, 1.00, 1.00, 1.00]],
    dtype=torch.double,
)

print("=== F6 Week 12 Configuration ===")
print(f"Kernel: Matérn-{KERNEL_NU} ARD (d={ARD_NUM_DIMS})")
print(f"Noise floor: {NOISE_LB} (was 1e-2)")
print(f"MLL restarts: {N_MLL_RESTARTS}")
print(f"Batch size q={Q_BATCH}")
print(f"Raw samples: {RAW_SAMPLES} (was 3000)")
print(f"Acquisition restarts: {NUM_RESTARTS}")
print(f"MC samples: {MC_SAMPLES}")
print(f"Interior penalty: STEEPNESS={STEEPNESS}, FLOOR={FLOOR}")
print(f"Milk threshold: ≥{MILK_THRESHOLD} (fallback: {MILK_FALLBACK})")
print(f"Bounds: {BOUNDS[0].tolist()} to {BOUNDS[1].tolist()}")

In [ ]:
# ── Data Preparation ──────────────────────────────────────────────────────────
X_train = torch.tensor(inputs, dtype=torch.float64)
Y_train = torch.tensor(outputs, dtype=torch.float64).unsqueeze(-1)

print(f"=== Data Preparation ===")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"Output range: [{outputs.min():.6f}, {outputs.max():.6f}]")
print(f"All negative: {(outputs < 0).all()}")
print(f"Transform: Standardize(m=1) only (no log)")
print(f"\nPer-ingredient ranges:")
for i, name in enumerate(ingredient_names):
    print(f"  x{i} ({name:>6s}): [{inputs[:, i].min():.6f}, {inputs[:, i].max():.6f}]")

In [ ]:
# ── GP Fitting: 15-restart MLL with Matérn-1.5 + Standardize(m=1) ─────────────
best_model = None
best_loss = float("inf")

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="The input matches the stored training data")

print(f"\n{'Restart':>8} {'Neg MLL':>12}")
print("-" * 22)

for seed in range(N_MLL_RESTARTS):
    torch.manual_seed(seed)
    
    likelihood = GaussianLikelihood(noise_constraint=GreaterThan(NOISE_LB))
    covar = ScaleKernel(MaternKernel(nu=KERNEL_NU, ard_num_dims=ARD_NUM_DIMS))
    model = SingleTaskGP(X_train, Y_train, covar_module=covar, likelihood=likelihood,
                         outcome_transform=Standardize(m=1))
    
    model.covar_module.base_kernel.lengthscale = 0.5
    model.likelihood.noise = 0.1 * Y_train.var().item()
    model.covar_module.outputscale = 1.0
    
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    try:
        fit_gpytorch_mll(mll)
    except Exception:
        print(f"{seed:>8d} {'FAILED':>12}")
        continue
    
    model.eval()
    with torch.no_grad():
        output = model(X_train)
        loss = -mll(output, model.train_targets).item()
    
    print(f"{seed:>8d} {loss:>12.4f}")
    
    if loss < best_loss:
        best_loss = loss
        best_model = copy.deepcopy(model)

warnings.filterwarnings("default", category=RuntimeWarning)

assert best_model is not None, "All MLL restarts failed!"
best_model.eval()

ls = best_model.covar_module.base_kernel.lengthscale.detach().cpu().numpy().ravel()
noise = best_model.likelihood.noise.detach().item()
outputscale = best_model.covar_module.outputscale.detach().item()

print(f"\n{'='*50}")
print(f"Best MLL loss: {best_loss:.6f}")
print(f"{'='*50}")
print(f"Fitted Hyperparameters (Matérn-{KERNEL_NU} ARD):")
for i in range(ARD_NUM_DIMS):
    print(f"  ℓ_{i} ({ingredient_names[i]:>6s}): {ls[i]:.6f}")
print(f"  Output scale: {outputscale:.6f}")
print(f"  Noise:        {noise:.6f}")
print(f"  Noise ≥ {NOISE_LB}: {'✓' if noise >= NOISE_LB else '✗'}")

In [ ]:
# ── Acquisition + Rank-Based Interior Penalty + Milk Feasibility ──────────────
import numpy as np

best_model.eval()
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]))
acqf = qLogNoisyExpectedImprovement(
    model=best_model,
    X_baseline=X_train,
    sampler=sampler,
    prune_baseline=True,
)

candidates, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=BOUNDS,
    q=Q_BATCH,
    num_restarts=NUM_RESTARTS,
    raw_samples=RAW_SAMPLES,
)

candidates = torch.clamp(candidates, 0.0, 0.999999)

# Posterior means (original space via Standardize auto-untransform)
with torch.no_grad():
    posterior = best_model.posterior(candidates)
    pred_means = posterior.mean.squeeze(-1).cpu().numpy()

# Distances to training data
dists = torch.cdist(candidates, X_train).min(dim=1).values

print(f"=== qLogNEI Acquisition Results (q={Q_BATCH}) ===")
print(f"Acquisition value: {acq_value.item():.6f}")
print(f"{'Cand':>5} {'Coords':>55} {'Mean':>12} {'MinDist':>8}")
print("-" * 85)
for i in range(Q_BATCH):
    c = candidates[i].cpu().numpy()
    coord_str = "[" + ", ".join(f"{v:.6f}" for v in c) + "]"
    print(f"{i+1:>5} {coord_str:>55} {pred_means[i]:>12.6f} {dists[i].item():>8.4f}")

# Check milk values
x4_values = candidates[:, 4].cpu().numpy()
print(f"\nMilk (x4) values: {[f'{v:.4f}' for v in x4_values]}")

# ── Interior Penalty: w(x) = FLOOR + (1-FLOOR) * ∏ sin(πx_i)^(2·STEEPNESS) ──
cands_np = candidates.cpu().numpy()
interior_weight = FLOOR + (1 - FLOOR) * np.prod(
    np.sin(np.pi * cands_np) ** (2 * STEEPNESS), axis=1
)

print(f"\n=== Interior Penalty Weights (STEEPNESS={STEEPNESS}, FLOOR={FLOOR}) ===")
for i in range(Q_BATCH):
    label = "interior" if interior_weight[i] >= 0.5 else "boundary"
    print(f"  Cand {i+1}: w={interior_weight[i]:.6f} ({label})")

# ── Rank-Based Re-Scoring ──
rank_mean   = np.argsort(np.argsort(pred_means)) + 1
rank_weight = np.argsort(np.argsort(interior_weight)) + 1
combined_score = rank_mean + rank_weight

# Milk feasibility filter (≥0.12, fallback to ≥0.10)
milk_ok = x4_values >= MILK_THRESHOLD
if not np.any(milk_ok):
    print(f"\n⚠ No candidates with milk ≥ {MILK_THRESHOLD}, falling back to ≥ {MILK_FALLBACK}")
    milk_ok = x4_values >= MILK_FALLBACK
    milk_note = f"fallback milk ≥ {MILK_FALLBACK}"
else:
    milk_note = f"milk ≥ {MILK_THRESHOLD}"

feasible_indices = np.where(milk_ok)[0]

# From feasible candidates, pick highest combined score; tiebreak by distance
if len(feasible_indices) == 0:
    best_idx = int(np.argmax(combined_score))
    sel_reason = "fallback — highest combined score (no feasible)"
else:
    feasible_scores = combined_score[feasible_indices]
    max_score = feasible_scores.max()
    tied = feasible_indices[feasible_scores == max_score]
    if len(tied) == 1:
        best_idx = tied[0]
    else:
        tied_dists = dists[tied].cpu().numpy()
        best_idx = tied[np.argmax(tied_dists)]
    sel_reason = f"highest rank score among feasible ({milk_note}), tiebreak by distance"

x_new = cands_np[best_idx]
x_new = np.clip(x_new, 0.0, 0.999999)

proposed_query = "-".join(f"{v:.6f}" for v in x_new)

# Duplicate check
is_duplicate = False
for i in range(len(inputs)):
    if np.allclose(x_new, inputs[i], atol=1e-6):
        is_duplicate = True
        break

print(f"\n=== Rank-Based Selection ===")
print(f"{'Cand':>5} {'R(mean)':>8} {'R(w)':>6} {'Score':>6} {'Milk':>8} {'Feas':>5}")
print("-" * 40)
for i in range(Q_BATCH):
    feas = "✓" if milk_ok[i] else "✗"
    sel = " ◄" if i == best_idx else "  "
    print(f"{i+1:>5} {rank_mean[i]:>8d} {rank_weight[i]:>6d} {combined_score[i]:>6d} {x4_values[i]:>8.4f} {feas:>5}{sel}")

print(f"\n→ Selected candidate #{best_idx + 1} ({sel_reason})")
print(f"  Posterior mean: {pred_means[best_idx]:.6f}")
print(f"  Interior weight: {interior_weight[best_idx]:.6f}")
print(f"  Milk (x4): {x_new[4]:.6f}")
print(f"  Duplicate: {is_duplicate}")
print(f"\n>>> SUBMISSION: {proposed_query}")

In [ ]:
# ── Three-Panel Surrogate Visualisation + Dimension Relevance ─────────────────
sorted_dims = np.argsort(ls)
top2 = sorted_dims[:2]
fix_dims = sorted_dims[2:]

print(f"Top-2 dims: x{top2[0]} ({ingredient_names[top2[0]]}), x{top2[1]} ({ingredient_names[top2[1]]})")
print(f"Fixed dims: " + ", ".join(f"x{d} ({ingredient_names[d]})={x_new[d]:.4f}" for d in fix_dims))

g0 = np.linspace(0, 1, GRID_RES)
g1 = np.linspace(0, 1, GRID_RES)
G0, G1 = np.meshgrid(g0, g1)

grid_pts = np.tile(x_new, (GRID_RES * GRID_RES, 1))
grid_pts[:, top2[0]] = G0.ravel()
grid_pts[:, top2[1]] = G1.ravel()

grid_tensor = torch.tensor(grid_pts, dtype=torch.double)
best_model.eval()
with torch.no_grad():
    posterior = best_model.posterior(grid_tensor)
    grid_mu = posterior.mean.squeeze(-1).cpu().numpy().reshape(GRID_RES, GRID_RES)
    grid_sigma = posterior.variance.sqrt().squeeze(-1).cpu().numpy().reshape(GRID_RES, GRID_RES)

X_initial = inputs[:N_INITIAL]
X_submissions = inputs[N_INITIAL:]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

c1 = axes[0].contourf(G0, G1, grid_mu, levels=30, cmap="viridis")
axes[0].scatter(X_initial[:, top2[0]], X_initial[:, top2[1]], c="tab:blue", edgecolors="white", s=40, zorder=5, label="Initial")
axes[0].scatter(X_submissions[:, top2[0]], X_submissions[:, top2[1]], c="tab:orange", edgecolors="white", s=40, zorder=5, label="Submissions")
axes[0].scatter(x_new[top2[0]], x_new[top2[1]], c="tab:green", marker="*", s=200, zorder=6, label="Proposed")
axes[0].set_title(f"GP Posterior Mean (x{top2[0]} vs x{top2[1]})")
axes[0].set_xlabel(f"x{top2[0]} ({ingredient_names[top2[0]]})")
axes[0].set_ylabel(f"x{top2[1]} ({ingredient_names[top2[1]]})")
axes[0].legend(fontsize=7)
fig.colorbar(c1, ax=axes[0])

c2 = axes[1].contourf(G0, G1, grid_sigma, levels=30, cmap="magma")
axes[1].scatter(X_initial[:, top2[0]], X_initial[:, top2[1]], c="tab:blue", edgecolors="white", s=40, zorder=5)
axes[1].scatter(X_submissions[:, top2[0]], X_submissions[:, top2[1]], c="tab:orange", edgecolors="white", s=40, zorder=5)
axes[1].scatter(x_new[top2[0]], x_new[top2[1]], c="tab:green", marker="*", s=200, zorder=6)
axes[1].set_title(f"GP Posterior Std Dev (x{top2[0]} vs x{top2[1]})")
axes[1].set_xlabel(f"x{top2[0]} ({ingredient_names[top2[0]]})")
axes[1].set_ylabel(f"x{top2[1]} ({ingredient_names[top2[1]]})")
fig.colorbar(c2, ax=axes[1])

inv_ls = 1.0 / ls
inv_ls_norm = inv_ls / inv_ls.sum()
colors = ['darkorange' if d in top2 else 'steelblue' for d in range(DIM)]
axes[2].barh(range(DIM), inv_ls_norm, color=colors)
axes[2].set_yticks(range(DIM))
axes[2].set_yticklabels([f"x{j} ({ingredient_names[j]})" for j in range(DIM)])
axes[2].set_title("Dimension Relevance (1/ℓ, normalised)")
axes[2].set_xlabel("Relative Importance")

plt.suptitle("F6 — GP Matérn-1.5 ARD Surrogate (Week 12, Rank IP)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Convergence Plot ──────────────────────────────────────────────────────────
y_all = outputs.ravel()
running_best = np.maximum.accumulate(y_all)
obs_idx = np.arange(1, len(y_all) + 1)

plt.figure(figsize=(10, 5))
plt.plot(obs_idx, running_best, color="grey", linewidth=1.5, alpha=0.6, label="Running best")

n_subs = len(y_all) - N_INITIAL
plt.scatter(obs_idx[:N_INITIAL], y_all[:N_INITIAL], c="tab:blue", s=30, zorder=3, label=f"Initial ({N_INITIAL})")
plt.scatter(obs_idx[N_INITIAL:], y_all[N_INITIAL:], c="tab:orange", s=30, zorder=3, label=f"Submissions ({n_subs})")

plt.axvline(x=N_INITIAL + 0.5, color="red", linestyle="--", alpha=0.7, label="Initial → Submissions")
plt.axhline(y=pred_means[best_idx], color="tab:green", linestyle=":", alpha=0.7, label=f"Predicted: {pred_means[best_idx]:.4f}")

plt.xlabel("Observation Number")
plt.ylabel("Output")
plt.title("F6 — Convergence Plot (Week 12)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Running best at Week 12 end: {running_best[-1]:.6f}")
print(f"Predicted value for proposed point: {pred_means[best_idx]:.4f}")

# Final validation
print(f"\n=== Final Validation ===")
parts = proposed_query.split("-")
assert len(parts) == 5, f"Expected 5 dims, got {len(parts)}"
for p in parts:
    v = float(p)
    assert 0.0 <= v <= 0.999999, f"Value {v} out of bounds"
milk_val = float(parts[4])
print(f"✓ Format: {len(parts)} dimensions")
print(f"✓ Range: all in [0, 0.999999]")
print(f"✓ Milk (x4): {milk_val:.6f} {'≥ ' + str(MILK_THRESHOLD) if milk_val >= MILK_THRESHOLD else '≥ ' + str(MILK_FALLBACK) + ' (fallback)'}")
print(f"✓ Duplicate: {'NO' if not is_duplicate else 'YES ⚠'}")
print(f"\n>>> FINAL SUBMISSION: {proposed_query}")